In [4]:
# Fresh end-to-end hyperparameter search + training from scratch (no checkpoints, no saving)
# Model: HuBERT only
# Config: xlarge_hubert_stability.npy only
# Split: STRICT subject-wise 80/10/10 with fixed SEED
# Test: untouched during search; evaluated once at end (optional)

import itertools
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from tqdm import tqdm

# -------------------------
# 0) Fixed paths + seed
# -------------------------
SEED = 42

HUBERT_DIR = Path("/mnt/d/PhD/ALS_PROJECT1/ALS_Diagnosis_Meta/embeddings/Hubert")
X_PATH = HUBERT_DIR / "xlarge_hubert_stability.npy"
Y_PATH = HUBERT_DIR / "labels.npy"
SID_PATH = HUBERT_DIR / "subject_ids.npy"

assert X_PATH.exists(), f"Missing: {X_PATH}"
assert Y_PATH.exists(), f"Missing: {Y_PATH}"
assert SID_PATH.exists(), f"Missing: {SID_PATH}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# -------------------------
# 1) Load + encode labels
# -------------------------
X = np.load(X_PATH)  # float32 expected
y_raw = np.load(Y_PATH, allow_pickle=True)
subject_ids = np.load(SID_PATH, allow_pickle=True).astype(str)

assert len(X) == len(y_raw) == len(subject_ids), "Length mismatch X/y/subject_ids"

le = LabelEncoder()
y = le.fit_transform(y_raw)
n_classes = int(len(le.classes_))
print("n_classes =", n_classes, "classes =", list(map(str, le.classes_)))

# -------------------------
# 2) Subject-wise stratified 80/10/10 split
# -------------------------
uniq_subjects = np.unique(subject_ids)

# subject-level label = majority label for that subject
subj_y = np.zeros(len(uniq_subjects), dtype=int)
for i, sid in enumerate(uniq_subjects):
    ys = y[subject_ids == sid]
    vals, cnts = np.unique(ys, return_counts=True)
    subj_y[i] = int(vals[int(np.argmax(cnts))])

subj_train, subj_temp, ysubj_train, ysubj_temp = train_test_split(
    uniq_subjects, subj_y, test_size=0.2, stratify=subj_y, random_state=SEED
)
subj_val, subj_test, ysubj_val, ysubj_test = train_test_split(
    subj_temp, ysubj_temp, test_size=0.5, stratify=ysubj_temp, random_state=SEED
)

set_train = set(subj_train.tolist())
set_val = set(subj_val.tolist())
set_test = set(subj_test.tolist())
assert (set_train & set_val) == set()
assert (set_train & set_test) == set()
assert (set_val & set_test) == set()

train_idx = np.where(np.isin(subject_ids, list(set_train)))[0]
val_idx = np.where(np.isin(subject_ids, list(set_val)))[0]
test_idx = np.where(np.isin(subject_ids, list(set_test)))[0]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("subjects train/val/test:", len(set_train), len(set_val), len(set_test))
print("utterances train/val/test:", len(train_idx), len(val_idx), len(test_idx))
print("class counts train:", dict(zip(*np.unique(y_train, return_counts=True))))
print("class counts val  :", dict(zip(*np.unique(y_val, return_counts=True))))
print("class counts test :", dict(zip(*np.unique(y_test, return_counts=True))))

# -------------------------
# 3) Normalize using TRAIN stats only
# -------------------------
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0) + 1e-8

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

# -------------------------
# 4) Model + episodic training (same logic as your main)
# -------------------------
class ClassLSTM(nn.Module):
    def __init__(self, feature_dim, context_dim):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=context_dim,
            num_layers=1,
            batch_first=True,
        )
        self.fc = nn.Sequential(
            nn.Linear(context_dim, feature_dim),
            nn.Tanh(),
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        h_mean = output.mean(dim=1)
        return self.fc(h_mean)


class HyperMetaLearner(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        feature_dim,
        context_dim,
        num_classes,
        use_metric_learner=False,
        distance_metric="euclidean",
        distance_scale_init=10.0,
        dropout_p1=0.3,
        dropout_p2=0.2,
    ):
        super().__init__()
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.use_metric_learner = use_metric_learner
        self.distance_metric = distance_metric

        self.feature_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_p2),
            nn.Linear(hidden_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.Tanh(),
        )

        self.class_lstm = ClassLSTM(feature_dim=feature_dim, context_dim=context_dim)

        self.metric_learner = (
            nn.Sequential(
                nn.Linear(feature_dim, feature_dim),
                nn.ReLU(),
                nn.Linear(feature_dim, feature_dim),
            )
            if use_metric_learner
            else nn.Identity()
        )

        self.distance_scale = nn.Parameter(torch.tensor(float(distance_scale_init)))

    def forward(self, x):
        return self.feature_net(x)

    def compute_prototypes(self, support, support_labels, n_way):
        device_ = next(self.parameters()).device
        support_features = self.feature_net(support)
        prototypes = torch.zeros(n_way, self.feature_dim).to(device_)
        for cls in range(n_way):
            class_mask = support_labels == cls
            class_features = support_features[class_mask]
            if len(class_features) > 0:
                class_features = class_features.unsqueeze(0)
                prototypes[cls] = self.class_lstm(class_features).squeeze(0)
        return prototypes

    def compute_distances(self, prototypes, query_features):
        prototypes = prototypes + self.metric_learner(prototypes)
        query_features = query_features + self.metric_learner(query_features)

        if self.distance_metric == "euclidean":
            return torch.cdist(query_features, prototypes, p=2) * self.distance_scale
        if self.distance_metric == "manhattan":
            return torch.cdist(query_features, prototypes, p=1) * self.distance_scale
        if self.distance_metric == "sqeuclidean":
            q = query_features.unsqueeze(1)
            p = prototypes.unsqueeze(0)
            return ((q - p) ** 2).sum(dim=-1) * self.distance_scale
        if self.distance_metric == "cosine":
            q = F.normalize(query_features, p=2, dim=-1)
            p = F.normalize(prototypes, p=2, dim=-1)
            return (1 - torch.matmul(q, p.T)) * self.distance_scale
        raise ValueError(f"Unsupported distance metric: {self.distance_metric}")


def prototypical_loss(model, support, support_labels, query, query_labels, n_way):
    device_ = next(model.parameters()).device
    support, query = support.to(device_), query.to(device_)
    support_labels, query_labels = support_labels.to(device_), query_labels.to(device_)

    query_features = model(query)
    prototypes = model.compute_prototypes(support, support_labels, n_way)
    distances = model.compute_distances(prototypes, query_features)

    log_p_y = F.log_softmax(-distances, dim=1)
    loss = F.nll_loss(log_p_y, query_labels)
    _, preds = torch.max(log_p_y, 1)
    acc = torch.mean((preds == query_labels).float())
    probs = F.softmax(-distances, dim=1).detach().cpu().numpy()
    return loss, acc, preds, query_labels, probs


class BalancedTaskGenerator:
    def __init__(self, features, labels, n_way, k_shot, q_query):
        self.features = features
        self.labels = labels
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query

        self.class_indices = {}
        for idx, lab in enumerate(labels):
            if lab not in self.class_indices:
                self.class_indices[lab] = []
            self.class_indices[lab].append(idx)

        self.classes = list(self.class_indices.keys())

    def create_task(self):
        selected_classes = random.sample(self.classes, self.n_way)

        support_set = []
        query_set = []

        for cls in selected_classes:
            samples = random.sample(self.class_indices[cls], self.k_shot + self.q_query)
            support_set.extend([(samples[i], cls) for i in range(self.k_shot)])
            query_set.extend([(samples[i], cls) for i in range(self.k_shot, self.k_shot + self.q_query)])

        support_features = torch.stack([torch.FloatTensor(self.features[idx]) for idx, _ in support_set])
        support_labels = torch.LongTensor([lab for _, lab in support_set])

        query_features = torch.stack([torch.FloatTensor(self.features[idx]) for idx, _ in query_set])
        query_labels = torch.LongTensor([lab for _, lab in query_set])

        return support_features, support_labels, query_features, query_labels


def calculate_auc(all_probs, all_labels, n_classes):
    if n_classes == 2:
        if all_probs.ndim == 2 and all_probs.shape[1] == 2:
            all_probs = all_probs[:, 1]
        elif all_probs.ndim == 2 and all_probs.shape[1] == 1:
            all_probs = all_probs.ravel()
        auc = roc_auc_score(all_labels, all_probs)
        return float(auc), [float(auc)]
    bin_labels = label_binarize(all_labels, classes=range(n_classes))
    aucs = []
    for i in range(n_classes):
        aucs.append(float(roc_auc_score(bin_labels[:, i], all_probs[:, i])))
    return float(np.mean(aucs)), aucs


def episodic_val_acc(model, task_gen, n_way, meta_batch_size):
    device_ = next(model.parameters()).device
    model.eval()
    acc_sum = 0.0
    with torch.no_grad():
        for _ in range(meta_batch_size):
            support, support_labels, query, query_labels = task_gen.create_task()
            support = support.to(device_)
            query = query.to(device_)
            support_labels = support_labels.to(device_)
            query_labels = query_labels.to(device_)
            _, acc, *_ = prototypical_loss(model, support, support_labels, query, query_labels, n_way=n_way)
            acc_sum += float(acc.item())
    return acc_sum / meta_batch_size


def evaluate_test(model, task_gen, n_way, meta_batch_size):
    device_ = next(model.parameters()).device
    model.eval()
    test_acc = 0.0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for _ in range(meta_batch_size * 2):
            support, support_labels, query, query_labels = task_gen.create_task()
            support = support.to(device_)
            query = query.to(device_)
            support_labels = support_labels.to(device_)
            query_labels = query_labels.to(device_)

            _, acc, preds, true_labels, probs = prototypical_loss(
                model, support, support_labels, query, query_labels, n_way=n_way
            )
            test_acc += float(acc.item())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(true_labels.cpu().numpy().tolist())
            all_probs.extend(probs)

    test_acc /= (meta_batch_size * 2)
    all_preds = np.asarray(all_preds)
    all_labels = np.asarray(all_labels)
    all_probs = np.asarray(all_probs)
    return float(test_acc), all_preds, all_labels, all_probs


def train_from_scratch(
    *,
    X_train,
    y_train,
    X_val,
    y_val,
    n_way,
    k_shot,
    q_query,
    model_cfg,
    train_cfg,
):
    set_seed(SEED)

    train_task_gen = BalancedTaskGenerator(X_train, y_train, n_way=n_way, k_shot=k_shot, q_query=q_query)
    val_task_gen = BalancedTaskGenerator(X_val, y_val, n_way=n_way, k_shot=k_shot, q_query=q_query)

    model = HyperMetaLearner(
        input_dim=int(X_train.shape[1]),
        hidden_dim=model_cfg["hidden_dim"],
        feature_dim=model_cfg["feature_dim"],
        context_dim=model_cfg["context_dim"],
        num_classes=n_way,
        use_metric_learner=model_cfg["use_metric_learner"],
        distance_metric=model_cfg["distance_metric"],
        distance_scale_init=model_cfg["distance_scale_init"],
        dropout_p1=model_cfg["dropout_p1"],
        dropout_p2=model_cfg["dropout_p2"],
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg["lr"], weight_decay=train_cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "max", patience=train_cfg["scheduler_patience"], factor=train_cfg["scheduler_factor"]
    )

    best_val = -1.0
    best_epoch = -1
    best_state = None
    patience_ctr = 0

    for epoch in range(1, train_cfg["num_epochs"] + 1):
        model.train()
        for _ in tqdm(range(train_cfg["meta_batch_size"]), desc=f"epoch {epoch}", leave=False):
            support, support_labels, query, query_labels = train_task_gen.create_task()
            optimizer.zero_grad()
            loss, _, _, _, _ = prototypical_loss(model, support, support_labels, query, query_labels, n_way=n_way)

            # Match your main behavior: optional extra explicit L2 term
            if train_cfg["explicit_l2"]:
                device_ = next(model.parameters()).device
                l2_reg = torch.tensor(0.0, device=device_)
                for p in model.parameters():
                    l2_reg += torch.norm(p, p=2)
                loss = loss + train_cfg["weight_decay"] * l2_reg

            loss.backward()
            optimizer.step()

        val_acc = episodic_val_acc(model, val_task_gen, n_way=n_way, meta_batch_size=train_cfg["meta_batch_size"])
        scheduler.step(val_acc)

        if val_acc > best_val:
            best_val = float(val_acc)
            best_epoch = int(epoch)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= train_cfg["patience"]:
                break

    return best_val, best_epoch, best_state


# -------------------------
# 5) Hyperparameter search space
# -------------------------
# You asked to tune k/q AND dimensions. Keep this grid small to avoid overfitting val.
KQ_GRID = [(3, 3), (5, 5), (10, 10)]

# Dimensions to tune (examples; edit as you like)
DIMS_GRID = [
    (512, 256, 64),  # (hidden_dim, feature_dim, context_dim)
    (256, 128, 64),
]

# Metric learner + distance
METRIC_GRID = [
    (False, "euclidean"),
    (True, "cosine"),
]

# Training hyperparameters to tune
LR_GRID = [6e-5, 3e-5, 1e-5]
WD_GRID = [1e-4, 0.0]

# Fixed training knobs (early stopping used only for selection)
TRAIN_FIXED = dict(
    num_epochs=200,
    meta_batch_size=16,
    patience=20,
    scheduler_patience=10,
    scheduler_factor=0.5,
    explicit_l2=True,  # matches your main behavior
)

# Fixed model knobs (not tuning dropouts here; keep main defaults)
MODEL_FIXED = dict(
    dropout_p1=0.3,
    dropout_p2=0.2,
    distance_scale_init=10.0,
)

# Episode feasibility check (avoid random.sample errors)
def min_class_count(y_arr: np.ndarray) -> int:
    _, cnts = np.unique(y_arr, return_counts=True)
    return int(cnts.min())

min_train = min_class_count(y_train)
min_val = min_class_count(y_val)
print("min per-class train:", min_train, "min per-class val:", min_val)

best = {
    "val_acc": -1.0,
    "best_epoch": None,
    "k_shot": None,
    "q_query": None,
    "hidden_dim": None,
    "feature_dim": None,
    "context_dim": None,
    "use_metric_learner": None,
    "distance_metric": None,
    "lr": None,
    "weight_decay": None,
    "state_dict": None,
}

trials = 0
for (k_shot, q_query), (hidden_dim, feature_dim, context_dim), (use_metric_learner, distance_metric), lr, wd in itertools.product(
    KQ_GRID, DIMS_GRID, METRIC_GRID, LR_GRID, WD_GRID
):
    if (k_shot + q_query) > min_train or (k_shot + q_query) > min_val:
        continue

    trials += 1
    print(
        f"[trial {trials}] k={k_shot} q={q_query} "
        f"hd={hidden_dim} fd={feature_dim} cd={context_dim} "
        f"metric={use_metric_learner} dist={distance_metric} "
        f"lr={lr} wd={wd}",
        flush=True,
    )

    model_cfg = dict(
        hidden_dim=hidden_dim,
        feature_dim=feature_dim,
        context_dim=context_dim,
        use_metric_learner=use_metric_learner,
        distance_metric=distance_metric,
        **MODEL_FIXED,
    )
    train_cfg = dict(lr=lr, weight_decay=wd, **TRAIN_FIXED)

    val_acc, best_epoch, state = train_from_scratch(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        n_way=n_classes,
        k_shot=k_shot,
        q_query=q_query,
        model_cfg=model_cfg,
        train_cfg=train_cfg,
    )

    if val_acc > best["val_acc"]:
        best.update(
            dict(
                val_acc=float(val_acc),
                best_epoch=int(best_epoch),
                k_shot=int(k_shot),
                q_query=int(q_query),
                hidden_dim=int(hidden_dim),
                feature_dim=int(feature_dim),
                context_dim=int(context_dim),
                use_metric_learner=bool(use_metric_learner),
                distance_metric=str(distance_metric),
                lr=float(lr),
                weight_decay=float(wd),
                state_dict=state,
            )
        )

print("\n== Best hyperparameters (selected by VAL episodic acc; test untouched during search) ==")
print({k: v for k, v in best.items() if k != "state_dict"})

# -------------------------
# 6) Evaluate once on TEST with the selected best config (optional but recommended)
# -------------------------
EVAL_TEST_AT_END = True

if EVAL_TEST_AT_END:
    model_cfg = dict(
        hidden_dim=best["hidden_dim"],
        feature_dim=best["feature_dim"],
        context_dim=best["context_dim"],
        use_metric_learner=best["use_metric_learner"],
        distance_metric=best["distance_metric"],
        **MODEL_FIXED,
    )

    final_model = HyperMetaLearner(
        input_dim=int(X_train.shape[1]),
        hidden_dim=model_cfg["hidden_dim"],
        feature_dim=model_cfg["feature_dim"],
        context_dim=model_cfg["context_dim"],
        num_classes=n_classes,
        use_metric_learner=model_cfg["use_metric_learner"],
        distance_metric=model_cfg["distance_metric"],
        distance_scale_init=model_cfg["distance_scale_init"],
        dropout_p1=model_cfg["dropout_p1"],
        dropout_p2=model_cfg["dropout_p2"],
    ).to(device)
    final_model.load_state_dict(best["state_dict"], strict=True)

    test_task_gen = BalancedTaskGenerator(
        X_test, y_test, n_way=n_classes, k_shot=best["k_shot"], q_query=best["q_query"]
    )
    test_acc, all_preds, all_labels, all_probs = evaluate_test(
        final_model, test_task_gen, n_way=n_classes, meta_batch_size=TRAIN_FIXED["meta_batch_size"]
    )

    macro_f1 = float(f1_score(all_labels, all_preds, average="macro"))
    macro_auc, class_aucs = calculate_auc(all_probs, all_labels, n_classes)

    print("\n== Test (evaluated once after selecting best by VAL) ==")
    print("test_acc :", test_acc)
    print("macro_f1 :", macro_f1)
    print("macro_auc:", macro_auc)
    print("class_aucs:", class_aucs)

print("\nRecommended fixed epochs (from best val epoch):", best["best_epoch"])
print("Note: if you later train on train+val, you can start with this epoch count and adjust if needed.")


device = cuda
n_classes = 3 classes = ['0', '1', '2']
subjects train/val/test: 50 6 7
utterances train/val/test: 5493 662 771
class counts train: {np.int64(0): np.int64(1768), np.int64(1): np.int64(2285), np.int64(2): np.int64(1440)}
class counts val  : {np.int64(0): np.int64(226), np.int64(1): np.int64(324), np.int64(2): np.int64(112)}
class counts test : {np.int64(0): np.int64(218), np.int64(1): np.int64(331), np.int64(2): np.int64(222)}
min per-class train: 1440 min per-class val: 112
[trial 1] k=3 q=3 hd=512 fd=256 cd=64 metric=False dist=euclidean lr=6e-05 wd=0.0001


[trial 2] k=3 q=3 hd=512 fd=256 cd=64 metric=False dist=euclidean lr=6e-05 wd=0.0


[trial 3] k=3 q=3 hd=512 fd=256 cd=64 metric=False dist=euclidean lr=3e-05 wd=0.0001


[trial 4] k=3 q=3 hd=512 fd=256 cd=64 metric=False dist=euclidean lr=3e-05 wd=0.0


[trial 5] k=3 q=3 hd=512 fd=256 cd=64 metric=False dist=euclidean lr=1e-05 wd=0.0001


...

[trial 69] k=10 q=10 hd=256 fd=128 cd=64 metric=True dist=cosine lr=3e-05 wd=0.0001


[trial 70] k=10 q=10 hd=256 fd=128 cd=64 metric=True dist=cosine lr=3e-05 wd=0.0


[trial 71] k=10 q=10 hd=256 fd=128 cd=64 metric=True dist=cosine lr=1e-05 wd=0.0001


[trial 72] k=10 q=10 hd=256 fd=128 cd=64 metric=True dist=cosine lr=1e-05 wd=0.0



== Best hyperparameters (selected by VAL episodic acc; test untouched during search) ==
{'val_acc': 0.883333396166563, 'best_epoch': 56, 'k_shot': 5, 'q_query': 5, 'hidden_dim': 512, 'feature_dim': 256, 'context_dim': 64, 'use_metric_learner': True, 'distance_metric': 'cosine', 'lr': 3e-05, 'weight_decay': 0.0}

== Test (evaluated once after selecting best by VAL) ==
test_acc : 0.7687500454485416
macro_f1 : 0.7676467323526147
macro_auc: 0.9083723958333333
class_aucs: [0.9373828125, 0.9315625, 0.8561718749999999]

Recommended fixed epochs (from best val epoch): 56
Note: if you later train on train+val, you can start with this epoch count and adjust if needed.
